<a href="https://colab.research.google.com/github/ArshiaGarg11/self-pruning-neural-network/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# SELF-PRUNING NEURAL NETWORK (Tredence Case Study)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt

# ---------------- CONFIG ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 40
BATCH_SIZE = 128
LR = 1e-3

LAMBDA_LIST = [1e-4, 5e-4, 1e-3]

THRESHOLD = 1e-2
TEMP = 20.0


# ---------------- PRUNABLE LAYER ----------------
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_features))

        # learnable gates
        self.gate_scores = nn.Parameter(torch.zeros(out_features, in_features))

    def forward(self, x):
        gates = torch.sigmoid(TEMP * self.gate_scores)
        pruned_weights = self.weight * gates
        return F.linear(x, pruned_weights, self.bias)

    def get_gates(self):
        return torch.sigmoid(TEMP * self.gate_scores)


# ---------------- MODEL ----------------
class SelfPruningNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            PrunableLinear(3072, 1024),
            nn.ReLU(),
            PrunableLinear(1024, 512),
            nn.ReLU(),
            PrunableLinear(512, 256),
            nn.ReLU(),
            PrunableLinear(256, 10)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)

    def sparsity_loss(self):
        total, count = 0.0, 0

        for m in self.modules():
            if isinstance(m, PrunableLinear):
                g = m.get_gates()
                total += g.sum()
                count += g.numel()

        return total / count

    def collect_gates(self):
        gates = []
        for m in self.modules():
            if isinstance(m, PrunableLinear):
                gates.append(m.get_gates().detach().cpu().numpy().flatten())
        return np.concatenate(gates)


# ---------------- DATA ----------------
def get_data():
    transform = T.Compose([
        T.ToTensor(),
        T.Normalize((0.5,), (0.5,))
    ])

    train = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=transform)
    test = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=transform)

    train_loader = torch.utils.data.DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test, batch_size=256)

    return train_loader, test_loader


# ---------------- TRAIN ----------------
def train_model(lam, train_loader):
    model = SelfPruningNet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        model.train()
        correct, total = 0, 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()

            out = model(x)

            ce_loss = F.cross_entropy(out, y)
            sp_loss = model.sparsity_loss()

            loss = ce_loss + lam * sp_loss

            loss.backward()
            optimizer.step()

            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

        if epoch % 5 == 0:
            print(f"[λ={lam}] Epoch {epoch} | Acc={100*correct/total:.2f}%")

    return model


# ---------------- EVAL ----------------
def evaluate(model, test_loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    acc = 100 * correct / total

    gates = model.collect_gates()
    sparsity = (gates < THRESHOLD).mean() * 100

    return acc, sparsity, gates


# ---------------- MAIN ----------------
def main():
    train_loader, test_loader = get_data()

    results = []

    for lam in LAMBDA_LIST:
        print("\n========================")
        print(f"Lambda = {lam}")
        print("========================")

        model = train_model(lam, train_loader)
        acc, sp, gates = evaluate(model, test_loader)

        results.append((lam, acc, sp, gates))

        print(f"Test Acc: {acc:.2f}% | Sparsity: {sp:.2f}%")

    print("\nFINAL RESULTS")
    print("Lambda\tAccuracy\tSparsity")

    for lam, acc, sp, _ in results:
        print(f"{lam}\t{acc:.2f}\t\t{sp:.2f}")

    # Plot best model
    best = max(results, key=lambda x: x[1])
    gates = best[3]

    plt.hist(gates, bins=100)
    plt.axvline(THRESHOLD, linestyle='--')
    plt.yscale("log")
    plt.title("Gate Distribution")
    plt.show()


if __name__ == "__main__":
    main()